In [1]:
from google.colab import drive
drive.mount('/content/drive')
from pprint import pprint


Mounted at /content/drive


In [2]:
!pip install PyMuPDF python-docx


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 24.8 MB/s eta 0:00:00


Task 1-  Reading files


In [3]:
import fitz                       # PyMuPDF
from docx import Document
import pandas as pd
import os


def extract_text_from_pdf(pdf_path):

    doc = fitz.open(pdf_path)

    pages = []

    for page in doc:
        pages.append(page.get_text())

    doc.close()

    return "\n".join(pages)


def extract_text_from_docx(docx_path):

    document = Document(docx_path)

    parts = []

    for para in document.paragraphs:
        parts.append(para.text)

    for table in document.tables:
        for row in table.rows:
            for cell in row.cells:
                parts.append(cell.text)

    return "\n".join(parts)


def extract_text_from_txt(txt_path):

    with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()


def extract_text(file_path):

    ext = os.path.splitext(file_path)[1].lower()

    if ext == ".pdf":
        return extract_text_from_pdf(file_path)

    elif ext == ".docx":
        return extract_text_from_docx(file_path)

    elif ext == ".txt":
        return extract_text_from_txt(file_path)

    else:
        return ""

In [5]:
RESUME_DIR = "/content/drive/MyDrive/Data/Resumes"

records = []

for file_name in sorted(os.listdir(RESUME_DIR)):

    if file_name.startswith("~$"):
        continue

    file_path = os.path.join(RESUME_DIR, file_name)
    text = extract_text(file_path)

    records.append({
        "file_name": file_name,
        "text": text,
        "n_words": len(text.split())
    })

resume_df = pd.DataFrame(records)
resume_df

,file_name,text,n_words
0,ADITYA SHARMA.docx,\nADITYA SHARMA\nSenior Data Engineer\nBengalu...,342
1,ANANYA IYER.docx,\nANANYA IYER\nData Scientist (Computer Vision...,306
2,ARJUN MEHTA.docx,"\nARJUN MEHTA\nData Analyst\nPune, India | arj...",328
3,DIVYA KRISHNAN.docx,\nDIVYA KRISHNAN\nData Engineer (Streaming)\nB...,307
4,KARTHIK MENON.docx,"\nKARTHIK MENON\nData Engineer\nKochi, India |...",277
5,KAVYA REDDY.docx,\nKAVYA REDDY\nSenior Data Scientist (NLP)\nHy...,361
6,NEHA VERMA.docx,\nNEHA VERMA\nData Scientist (ML + MLOps)\nBen...,314
7,PRIYA NAIR.docx,"\nPRIYA NAIR\nSenior Data Analyst\nBengaluru, ...",426
8,ROHAN KAPOOR.docx,"\nROHAN KAPOOR\nData Scientist\nGurgaon, India...",292
9,SAMEER KHAN.docx,"\nSAMEER KHAN\nJunior Data Scientist\nMumbai, ...",267


Task 2 - Load the JD

In [6]:
JD_PATH = "/content/drive/MyDrive/Data/JD/JD 2.txt"

jd_text = extract_text(JD_PATH)

jd_text = jd_text.strip()

print("chars :", len(jd_text))
print("words :", len(jd_text.split()))
print("-" * 60)
print(jd_text)

chars : 4441
words : 582
------------------------------------------------------------
## Title: Data Analyst – Business Intelligence

**Location:** Pune, India (Hybrid)
**Experience Required:** 2–5 Years
**Employment Type:** Full-Time

### About the Role

We are looking for a highly motivated **Data Analyst – Business Intelligence** to join our Analytics team. In this role, you will transform raw business data into meaningful insights that drive strategic decision-making across multiple business functions. You will collaborate with cross-functional teams, including Product, Marketing, Sales, Finance, and Operations, to understand business requirements, build insightful dashboards, perform statistical analyses, and provide actionable recommendations.

The ideal candidate is passionate about data, has strong analytical thinking, and enjoys solving complex business problems through data visualization and reporting. You should be comfortable working with large datasets, creating automated 

Task 3: Preprocessing (JD + resumes)

In [7]:
import re


def preprocess(text):

    if not isinstance(text, str):
        return ""

    # lowercase
    text = text.lower()

    # normalise bullets and tabs
    text = text.replace("\t", " ")
    text = re.sub(r"[•‣▪◦●·∙]", "-", text)

    # drop urls
    text = re.sub(r"http\S+|www\.\S+", " ", text)

    # keep letters, digits and the symbols that matter in tech terms
    # (c++, node.js, ci/cd, 3.5, r&d)
    text = re.sub(r"[^a-z0-9+#./\-,()&@:\n ]", " ", text)

    # tidy whitespace, but keep line breaks
    text = re.sub(r"[ ]{2,}", " ", text)
    text = "\n".join(line.strip() for line in text.split("\n"))
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


# apply to resumes
resume_df["clean_text"] = resume_df["text"].apply(preprocess)

# apply to JD
jd_clean = preprocess(jd_text)

print("JD  :", len(jd_clean.split()), "words")
print("RES :", resume_df["clean_text"].apply(lambda t: len(t.split())).tolist())
print("-" * 60)
print(jd_clean[:600])
print("-" * 60)
print(resume_df.loc[0, "clean_text"][:600])

JD  : 551 words
RES : [332, 292, 316, 294, 266, 350, 301, 416, 282, 257, 271, 283]
------------------------------------------------------------
## title: data analyst business intelligence

location: pune, india (hybrid)
experience required: 2 5 years
employment type: full-time

### about the role

we are looking for a highly motivated data analyst business intelligence to join our analytics team. in this role, you will transform raw business data into meaningful insights that drive strategic decision-making across multiple business functions. you will collaborate with cross-functional teams, including product, marketing, sales, finance, and operations, to understand business requirements, build insightful dashboards, perform statisti
------------------------------------------------------------
aditya sharma
senior data engineer
bengaluru, india aditya.sharma@email.com +91-90000-00009 linkedin.com/in/adityasharma

------------------------------------------------------------------------

Task 4: Split resume into sections



In [9]:

import re
import pandas as pd

# canonical section -> variants seen in real resumes
SECTION_HEADINGS = {
    "summary":        ["summary", "professional summary", "career summary", "profile",
                       "about me", "objective", "career objective"],
    "skills":         ["skills", "technical skills", "key skills", "core competencies",
                       "technical expertise", "technologies", "tech stack", "skill set"],
    "experience":     ["experience", "work experience", "professional experience",
                       "employment history", "work history", "career history", "employment"],
    "projects":       ["projects", "academic projects", "personal projects",
                       "key projects", "project experience"],
    "education":      ["education", "academic background", "academic qualifications",
                       "educational qualification", "qualifications"],
    "certifications": ["certifications", "certification", "courses", "training",
                       "licenses and certifications"],
    "achievements":   ["achievements", "awards", "accomplishments", "honors",
                       "publications", "extracurricular"],
}


def match_heading(line):

    line = line.strip().strip(":").strip("-").strip()

    # headings are short; a long line is body text
    if len(line.split()) > 5 or len(line) < 3:
        return None

    for section, variants in SECTION_HEADINGS.items():
        for v in variants:
            if line == v or line.startswith(v):
                return section

    return None


def split_into_sections(text):

    sections = {}
    current = "header"          # name / contact sits above the first heading
    buffer = []

    for line in text.split("\n"):

        heading = match_heading(line)

        if heading:
            if buffer:
                sections[current] = sections.get(current, "") + "\n" + "\n".join(buffer)
            current = heading
            buffer = []
        else:
            if line.strip():
                buffer.append(line.strip())

    if buffer:
        sections[current] = sections.get(current, "") + "\n" + "\n".join(buffer)

    sections = {k: v.strip() for k, v in sections.items() if v.strip()}

    # FALLBACK: no real headings found -> treat the whole resume as one block
    real = [k for k in sections if k != "header"]

    if len(real) < 2:
        return {"full_text": text.strip()}, True

    return sections, False


results = resume_df["clean_text"].apply(split_into_sections)

resume_df["sections"] = results.apply(lambda x: x[0])
resume_df["used_fallback"] = results.apply(lambda x: x[1])

# ---- what got found ----
tracked = ["skills", "experience", "projects", "education"]

summary = []

for _, row in resume_df.iterrows():
    found = row["sections"].keys()
    summary.append({
        "doc_id": row["doc_id"] if "doc_id" in row else row["file_name"],
        **{s: ("Y" if s in found else "-") for s in tracked},
        "n_sections": len(found),
        "fallback": "YES" if row["used_fallback"] else ""
    })


# ---- eyeball one ----
secs = resume_df.loc[0, "sections"]

resume_df['sections'][0]

{'header': 'aditya sharma\nsenior data engineer\nbengaluru, india aditya.sharma@email.com +91-90000-00009 linkedin.com/in/adityasharma\n--------------------------------------------------------------------------------',
 'summary': '--------------------------------------------------------------------------------\nsenior data engineer with 7 years of experience designing and operating large-scale\nbatch and streaming pipelines on aws. deep spark, airflow, and kafka expertise, with\nownership of data warehousing and pipeline reliability for analytics and ml teams.\ntrack record of cutting pipeline cost and runtime while improving data quality and\nfreshness at scale.\n--------------------------------------------------------------------------------',
 'skills': '--------------------------------------------------------------------------------\nbig data & streaming: apache spark, apache kafka, spark structured streaming,\ndatabricks\norchestration: apache airflow\ncloud (aws): s3, redshift, 

Task 5: Split the JD into sections

In [11]:

import re

JD_HEADING_MAP = {
    "role":             ["job title", "title", "role", "position", "designation",
                         "about the role", "role overview", "job summary",
                         "about the job", "overview"],
    "required_skills":  ["required skills", "requirements", "must have", "must haves",
                         "required qualifications", "skills required", "key skills",
                         "technical skills", "what we are looking for", "who you are",
                         "essential skills", "mandatory skills", "technical requirements"],
    "preferred_skills": ["preferred skills", "nice to have", "good to have", "bonus",
                         "preferred qualifications", "desirable", "plus points",
                         "added advantage"],
    "responsibilities": ["responsibilities", "key responsibilities", "what you will do",
                         "what you'll do", "duties", "job description", "your role",
                         "day to day"],
    "experience":       ["experience", "experience required", "work experience",
                         "years of experience", "eligibility"],
    "education":        ["education", "qualification", "qualifications",
                         "educational qualification", "academic requirements"],
    "company":          ["about us", "about the company", "who we are", "company overview"],
    "benefits":         ["benefits", "what we offer", "perks", "compensation", "salary"],
}

DROP_SECTIONS = ["company", "benefits"]


def clean_heading_candidate(line):
    """Strip markdown decoration: ## heading ##, **bold**, trailing colon."""
    s = line.strip()
    s = s.strip("#").strip()
    s = s.replace("*", "").replace("_", "")
    s = s.strip("-").strip("=").strip()
    s = s.strip(":").strip()
    return s.lower().strip()


def lookup_section(candidate):

    if len(candidate) < 3 or len(candidate.split()) > 6:
        return None

    for section, variants in JD_HEADING_MAP.items():
        for v in variants:
            if candidate == v or candidate.startswith(v):
                return section

    return None


def match_jd_heading(line):
    """Returns (section, inline_content).

    Handles 'experience: 2+ years' by returning ('experience', '2+ years')
    so the value survives instead of being eaten as a heading."""

    raw = line.strip()

    # separator lines like ---, ===, ***
    if not raw or set(raw) <= set("-=_#* "):
        return None, ""

    head, sep, rest = raw.partition(":")

    if sep:
        section = lookup_section(clean_heading_candidate(head))
        if section:
            return section, rest.strip()

    section = lookup_section(clean_heading_candidate(raw))
    if section:
        return section, ""

    return None, ""


def split_jd_into_sections(text):

    sections = {}
    current = "role"
    buffer = []

    def flush():
        if buffer:
            sections[current] = (sections.get(current, "") + "\n" + "\n".join(buffer)).strip()

    for line in text.split("\n"):

        section, inline = match_jd_heading(line)

        if section:
            flush()
            current = section
            buffer = [inline] if inline else []
        else:
            if line.strip() and not set(line.strip()) <= set("-=_#* "):
                buffer.append(line.strip())

    flush()

    sections = {k: v.strip() for k, v in sections.items() if v.strip()}

    for noise in DROP_SECTIONS:
        sections.pop(noise, None)

    if len([k for k in sections if k != "role"]) < 1:
        return {"full_text": text.strip()}, True

    return sections, False


jd_sections, jd_fallback = split_jd_into_sections(jd_clean)


In [12]:
pprint(jd_sections)

{'experience': '2 5 years\nemployment type: full-time',
 'preferred_skills': 'experience using python (pandas, numpy, matplotlib, '
                     'seaborn) for data analysis and automation.\n'
                     'familiarity with google analytics , looker , metabase , '
                     'or similar bi platforms.\n'
                     'knowledge of etl processes and data warehousing '
                     'concepts.\n'
                     'understanding of data modeling and dimensional modeling '
                     'techniques.\n'
                     'experience with cloud platforms such as aws, azure, or '
                     'google cloud platform.\n'
                     'basic knowledge of version control tools such as git.\n'
                     'exposure to machine learning concepts and predictive '
                     'analytics.\n'
                     'experience working in agile or scrum environments.',
 'required_skills': 'bachelor s degree in statistics

Task 6 : Chunk + embed JD and resumes

In [13]:

!pip install -q sentence-transformers

import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

MODEL_NAME   = "BAAI/bge-large-en-v1.5"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
MAX_WORDS    = 60

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(MODEL_NAME, device=device)
DIM = model.get_sentence_embedding_dimension()

print("model :", MODEL_NAME)
print("device:", device)
print("dim   :", DIM)


def chunk_text(text, max_words=MAX_WORDS):
    """Group lines into chunks under a word budget.
    Never splits a line, so a bullet stays whole and can be quoted
    as evidence in Task 9."""

    chunks, buffer, count = [], [], 0

    for line in text.split("\n"):

        line = line.strip()
        if not line:
            continue

        n = len(line.split())

        if count + n > max_words and buffer:
            chunks.append(" ".join(buffer))
            buffer, count = [], 0

        buffer.append(line)
        count += n

    if buffer:
        chunks.append(" ".join(buffer))

    return chunks


def embed(texts, is_query=False):
    """normalize_embeddings=True means cosine similarity is just a dot product."""

    if not texts:
        return np.zeros((0, DIM), dtype="float32")

    if is_query:
        texts = [QUERY_PREFIX + t for t in texts]

    return model.encode(texts,
                        batch_size=16,
                        normalize_embeddings=True,
                        convert_to_numpy=True,
                        show_progress_bar=False)


def embed_sections(sections, is_query=False):

    chunks = {s: chunk_text(t) for s, t in sections.items()}
    vecs   = {s: embed(c, is_query=is_query) for s, c in chunks.items()}

    return chunks, vecs


# ---- JD is the query side ----
jd_chunks, jd_vecs = embed_sections(jd_sections, is_query=True)

# ---- resumes are the passage side ----
all_chunks, all_vecs = [], []

for sections in tqdm(resume_df["sections"], desc="embedding resumes"):
    c, v = embed_sections(sections, is_query=False)
    all_chunks.append(c)
    all_vecs.append(v)

resume_df["chunks"] = all_chunks
resume_df["vecs"]   = all_vecs


# ---- summary ----
print("\nJD chunks per section:")
for s, c in jd_chunks.items():
    print(f"  {s:<18} {len(c)}")

print("\nResume chunks:")
for i, row in resume_df.iterrows():
    total = sum(len(c) for c in row["chunks"].values())
    print(f"  {row['file_name']:<35} {total:>3} chunks across {len(row['chunks'])} sections")


# ---- sanity check: does the similarity look sane? ----
if "required_skills" in jd_vecs and "skills" in resume_df.loc[0, "vecs"]:
    sim = jd_vecs["required_skills"] @ resume_df.loc[0, "vecs"]["skills"].T
    print("\nJD required_skills vs resume[0] skills")
    print("  shape:", sim.shape)
    print("  max  :", round(float(sim.max()), 4))
    print("  mean :", round(float(sim.mean()), 4))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

/tmp/ipykernel_1603/3170753332.py:14: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  DIM = model.get_sentence_embedding_dimension()


model : BAAI/bge-large-en-v1.5
device: cuda
dim   : 1024


embedding resumes:   0%|          | 0/12 [00:00<?, ?it/s]


JD chunks per section:
  role               3
  experience         1
  responsibilities   3
  required_skills    3
  preferred_skills   2

Resume chunks:
  ADITYA SHARMA.docx                    9 chunks across 8 sections
  ANANYA IYER.docx                      9 chunks across 8 sections
  ARJUN MEHTA.docx                      9 chunks across 8 sections
  DIVYA KRISHNAN.docx                   9 chunks across 8 sections
  KARTHIK MENON.docx                    9 chunks across 8 sections
  KAVYA REDDY.docx                      9 chunks across 8 sections
  NEHA VERMA.docx                       9 chunks across 8 sections
  PRIYA NAIR.docx                      11 chunks across 8 sections
  ROHAN KAPOOR.docx                     9 chunks across 8 sections
  SAMEER KHAN.docx                      9 chunks across 8 sections
  SNEHA PATIL.docx                      9 chunks across 8 sections
  VIKRAM RAO.docx                       9 chunks across 8 sections

JD required_skills vs resume[0] skills
 

In [14]:
(jd_chunks)

{'role': ['data analyst business intelligence location: pune, india (hybrid)',
  'we are looking for a highly motivated data analyst business intelligence to join our analytics team. in this role, you will transform raw business data into meaningful insights that drive strategic decision-making across multiple business functions. you will collaborate with cross-functional teams, including product, marketing, sales, finance, and operations, to understand business requirements, build insightful dashboards, perform statistical analyses, and provide actionable recommendations.',
  'the ideal candidate is passionate about data, has strong analytical thinking, and enjoys solving complex business problems through data visualization and reporting. you should be comfortable working with large datasets, creating automated reporting solutions, and communicating insights to both technical and non-technical stakeholders.'],
 'experience': ['2 5 years employment type: full-time'],
 'responsibilities

In [15]:
jd_vecs

{'role': array([[-0.03419791, -0.00442899, -0.02796768, ..., -0.04271627,
         -0.04127287,  0.01176691],
        [-0.00772833,  0.00899818, -0.04594128, ..., -0.03230547,
         -0.06428756,  0.01414159],
        [ 0.0079094 ,  0.0058334 ,  0.0137256 , ...,  0.01094645,
         -0.01785126, -0.00878048]], dtype=float32),
 'experience': array([[-0.01341433,  0.00362823,  0.06248248, ..., -0.02692905,
         -0.00888831, -0.01861822]], dtype=float32),
 'responsibilities': array([[-0.03745872,  0.0034126 , -0.02228042, ..., -0.03148436,
         -0.00058746,  0.0144452 ],
        [ 0.01599245,  0.01387622, -0.02016574, ..., -0.00403779,
         -0.02012113,  0.00034814],
        [-0.00137051, -0.00794747,  0.01093547, ..., -0.01363957,
          0.01261456, -0.00142935]], dtype=float32),
 'required_skills': array([[-0.00953116,  0.0132894 ,  0.01117421, ..., -0.05086753,
         -0.02347982,  0.03090772],
        [-0.00159541,  0.00815996,  0.0205925 , ..., -0.03059881,
      

Task 7a: Compare JD section embeddings with relevant resume section embeddings

In [16]:
import numpy as np
import pandas as pd

# which resume sections each JD section is allowed to match against,
# in priority order
SECTION_MAP = {
    "required_skills":  ["skills", "experience", "projects"],
    "experience":       ["experience", "summary"],
    "responsibilities": ["experience", "projects"],
    "preferred_skills": ["skills", "projects", "experience"],
}

TOP_EVIDENCE = 3


def compare_section(jd_vecs_sec, jd_chunks_sec, res_vecs, res_chunks, targets):
    """Max-pool each JD chunk against the resume, then average.

    Mean-of-max answers 'is every JD requirement covered somewhere?'
    Plain mean would punish a resume for containing anything off-topic."""

    # resume used the fallback -> only one block exists
    if "full_text" in res_vecs:
        targets = ["full_text"]

    mats, texts, srcs = [], [], []

    for t in targets:
        if t in res_vecs and len(res_vecs[t]) > 0:
            mats.append(res_vecs[t])
            texts.extend(res_chunks[t])
            srcs.extend([t] * len(res_chunks[t]))

    if not mats or len(jd_vecs_sec) == 0:
        return None, []

    R = np.vstack(mats)

    sim = jd_vecs_sec @ R.T            # (n_jd_chunks, n_resume_chunks)

    best_idx = sim.argmax(axis=1)
    best_val = sim.max(axis=1)

    score = float(best_val.mean())

    evidence = []
    for i in range(len(best_val)):
        j = int(best_idx[i])
        evidence.append({
            "jd_chunk":       jd_chunks_sec[i],
            "resume_chunk":   texts[j],
            "resume_section": srcs[j],
            "sim":            round(float(best_val[i]), 4),
        })

    evidence.sort(key=lambda e: e["sim"], reverse=True)

    return score, evidence[:TOP_EVIDENCE]


all_scores, all_evidence = [], []

for _, row in resume_df.iterrows():

    scores, evid = {}, {}

    for jd_sec, targets in SECTION_MAP.items():

        if jd_sec not in jd_vecs:
            continue

        s, e = compare_section(jd_vecs[jd_sec], jd_chunks[jd_sec],
                               row["vecs"], row["chunks"], targets)

        scores[jd_sec] = s
        evid[jd_sec] = e

    all_scores.append(scores)
    all_evidence.append(evid)

resume_df["section_scores"] = all_scores
resume_df["evidence"] = all_evidence


# ---- similarity matrix ----
matrix = pd.DataFrame(
    [{"file_name": r["file_name"], **{k: (round(v, 4) if v is not None else None)
                                      for k, v in r["section_scores"].items()}}
     for _, r in resume_df.iterrows()]
).set_index("file_name")

# ---- evidence for the current best candidate on required_skills ----
if "required_skills" in matrix.columns:

    top = matrix["required_skills"].idxmax()
    row = resume_df[resume_df["file_name"] == top].iloc[0]

    print(f"\n{'='*60}\nEVIDENCE — {top}\n{'='*60}")

    for jd_sec, items in row["evidence"].items():
        if not items:
            continue
        print(f"\n[{jd_sec}]")
        for e in items:
            print(f"  {e['sim']:.3f}  (from resume '{e['resume_section']}')")
            print(f"    JD     : {e['jd_chunk'][:110]}")
            print(f"    RESUME : {e['resume_chunk'][:110]}")


EVIDENCE — PRIYA NAIR.docx

[required_skills]
  0.724  (from resume 'skills')
    JD     : advanced knowledge of microsoft excel , including pivot tables, power query, power pivot, lookup functions, an
    RESUME : -------------------------------------------------------------------------------- querying & data: sql (advance
  0.703  (from resume 'skills')
    JD     : bachelor s degree in statistics, mathematics, computer science, economics, business analytics, or a related fi
    RESUME : -------------------------------------------------------------------------------- querying & data: sql (advance
  0.521  (from resume 'skills')
    JD     : ability to work with cross-functional teams in a fast-paced environment.
    RESUME : -------------------------------------------------------------------------------- querying & data: sql (advance

[experience]
  0.534  (from resume 'summary')
    JD     : 2 5 years employment type: full-time
    RESUME : -----------------------------------------

Task 8: Normalise score, rank, take top K

In [18]:
def calculate_raw_score(scores):
    valid_scores = [s for s in scores.values() if pd.notna(s)]
    return sum(valid_scores) / len(valid_scores) if valid_scores else float("nan")

resume_df["raw_score"] = resume_df["section_scores"].apply(calculate_raw_score)

In [20]:

import pandas as pd

TOP_K = 5

raw = resume_df["raw_score"]
lo, hi = raw.min(), raw.max()

# BGE similarities sit in a narrow band (~0.6-0.85), so a weak candidate
# would otherwise display as "72% match". Min-max spreads the pool across
# 0-100 for display; raw_score is kept for auditability.
if hi - lo < 1e-6:
    resume_df["fit_score"] = 50.0
else:
    resume_df["fit_score"] = ((raw - lo) / (hi - lo) * 100).round(1)

ranked = resume_df.sort_values("raw_score", ascending=False).reset_index(drop=True)
ranked["rank"] = ranked.index + 1

shortlist = ranked.head(TOP_K).copy()

print(f"Shortlist: top {TOP_K} of {len(ranked)}\n")
display(shortlist[["rank", "file_name", "fit_score", "raw_score"]])

Shortlist: top 5 of 12



,rank,file_name,fit_score,raw_score
0,1,PRIYA NAIR.docx,100.0,0.639889
1,2,ARJUN MEHTA.docx,89.9,0.629083
2,3,SAMEER KHAN.docx,58.6,0.595795
3,4,KARTHIK MENON.docx,52.4,0.589195
4,5,ROHAN KAPOOR.docx,49.1,0.585702


Task 9: NER + contact extraction

In [22]:

import re
import torch
import pandas as pd
from transformers import pipeline

NER_MODEL = "yashpwr/resume-ner-bert"

ner = pipeline("ner",
               model=NER_MODEL,
               aggregation_strategy="simple",
               device=0 if torch.cuda.is_available() else -1)

HEADER_LINES = 40      # name/contact live at the top; NER on the full resume is slow and noisy


def header_text(text, n=HEADER_LINES):
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    return "\n".join(lines[:n])[:1500]


def extract_contact(text):
    """Regex, not NER. Transformer NER is unreliable on emails and phone
    numbers because the tokeniser shreds them."""

    email = re.findall(r"[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}", text)

    phone = []
    for cand in re.findall(r"(?:\+?\d{1,3}[\s\-]?)?(?:\(?\d{2,5}\)?[\s\-]?)?\d{3,5}[\s\-]?\d{4,6}",
                           text):
        digits = re.sub(r"\D", "", cand)
        if 10 <= len(digits) <= 13:          # filters out years and ID numbers
            phone.append(cand.strip())

    linkedin = re.findall(r"linkedin\.com/in/[A-Za-z0-9\-_%]+", text, flags=re.I)
    github   = re.findall(r"github\.com/[A-Za-z0-9\-_]+", text, flags=re.I)

    return {
        "email":    email[0] if email else "",
        "phone":    phone[0] if phone else "",
        "linkedin": linkedin[0] if linkedin else "",
        "github":   github[0] if github else "",
    }


def run_ner(text):
    """Returns {entity_group: [values]} with duplicates removed."""

    try:
        ents = ner(text)
    except Exception as e:
        print("  NER failed:", e)
        return {}

    out = {}
    for e in ents:
        if e["score"] < 0.60:
            continue
        g = e["entity_group"]
        v = e["word"].replace(" ##", "").strip()
        if len(v) < 2:
            continue
        out.setdefault(g, [])
        if v not in out[g]:
            out[g].append(v)

    return out


def pick_name(ner_out, fallback_lines):
    """Take the NER name if there is one, else the first plausible header line."""

    for key in ner_out:
        if "NAME" in key.upper() or key.upper() in ("PER", "PERSON"):
            return ner_out[key][0].title()

    for line in fallback_lines.split("\n")[:5]:
        line = line.strip()
        if 1 < len(line.split()) <= 4 and "@" not in line and not re.search(r"\d{4}", line):
            return line.title()

    return ""


rows, all_ner = [], []

for _, r in shortlist.iterrows():

    # IMPORTANT: original cased text, not clean_text.
    # Task 5 lowercased everything, and lowercasing destroys BERT NER on names.
    original = r["text"]
    head = header_text(original)

    ents = run_ner(head)
    contact = extract_contact(original)

    all_ner.append(ents)

    rows.append({
    "rank": r["rank"],
    "name": pick_name(ents, head),
    "fit_score": r["fit_score"],
    **contact,
    "file_name": r["file_name"],
})

shortlist["ner"] = all_ner
candidates = pd.DataFrame(rows)

display(candidates)

# ---- what labels this model actually emits (useful for tuning pick_name) ----
labels = sorted({k for e in all_ner for k in e})
print("\nentity groups found:", labels)

print("\nfull NER output — rank 1:")
for k, v in all_ner[0].items():
    print(f"  {k:<16} {v[:6]}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,rank,name,fit_score,email,phone,linkedin,github,file_name
0,1,Priya Nair,100.0,priya.nair@email.com,+91-90000-00001,linkedin.com/in/priyanair,,PRIYA NAIR.docx
1,2,Arjun Mehta,89.9,arjun.mehta@email.com,+91-90000-00002,linkedin.com/in/arjunmehta,,ARJUN MEHTA.docx
2,3,Sameer Khan,58.6,sameer.khan@email.com,+91-90000-00006,linkedin.com/in/sameerkhan,,SAMEER KHAN.docx
3,4,Karthik Menon,52.4,karthik.menon@email.com,+91-90000-00011,linkedin.com/in/karthikmenon,,KARTHIK MENON.docx
4,5,Rohan Kapoor,49.1,rohan.kapoor@email.com,+91-90000-00004,linkedin.com/in/rohankapoor,,ROHAN KAPOOR.docx



entity groups found: ['Designation', 'Email Address']

full NER output — rank 1:
  Designation      ['Senior Data Analyst', 'Senior', 'Analyst']
  Email Address    ['priya. nair @ email. com', '. com /']
